In [1]:
# -*- coding: utf-8 -*-
"""Mock User Behavior Dataset Generator

This script generates a comprehensive mock dataset simulating user behavior
for analytics purposes. The dataset follows the 'analytics-ready' schema
and supports all planned analytics outcomes (CO2, CO4, CO5, CO6, CO7).

Author: Analytics Team
Date: 2024
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

print("🚀 Starting Mock Dataset Generation...")

# =============================================================================
# CONFIGURATION & PARAMETERS
# =============================================================================

# Time range for the dataset (last 90 days)
END_DATE = datetime.now()
START_DATE = END_DATE - timedelta(days=90)

# User base parameters
TOTAL_USERS = 1000
TOTAL_CONVERSATIONS = 5000
TOTAL_MESSAGES = 25000
TOTAL_EVENTS = 75000

# User segmentation parameters
USER_TYPES = ['power_user', 'regular_user', 'casual_user', 'churn_risk_user']
USER_TYPE_DISTRIBUTION = [0.15, 0.45, 0.30, 0.10]  # 15% power users, 10% churn risk

# Conversation categories
CONVERSATION_CATEGORIES = [
    'customer_support', 'product_inquiry', 'technical_help', 
    'billing_question', 'feature_request', 'general_feedback'
]

# Event types
EVENT_TYPES = [
    'app_launch', 'message_sent', 'conversation_started', 
    'conversation_ended', 'settings_changed', 'premium_feature_used',
    'notification_received', 'profile_updated', 'search_performed'
]

# Mock message templates
MESSAGE_TEMPLATES = [
    "Hello, I need help with {feature}.",
    "Can you explain how to use {feature}?",
    "I'm having trouble with {feature}.",
    "Is there a way to {action} in the app?",
    "Thank you for your help!",
    "That solved my problem.",
    "I have a question about my account.",
    "When will {feature} be available?",
    "How much does {service} cost?",
    "Can you help me with {issue}?",
    "I'd like to report a bug in {feature}.",
    "Is there documentation for {topic}?",
    "The app is working great!",
    "I have a suggestion for improvement.",
    "Could you add support for {request}?",
    "My subscription is not working.",
    "How do I cancel my account?",
    "I forgot my password.",
    "The performance has been slow recently.",
    "I love the new update!"
]

# Feature names for message templates
FEATURES = ['dashboard', 'messaging', 'billing', 'settings', 'profile', 'notifications', 'search', 'reports']
ACTIONS = ['export data', 'change settings', 'upgrade plan', 'invite friends', 'create report']
ISSUES = ['login', 'upload', 'download', 'sync', 'payment']

print("✅ Configuration loaded successfully")

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def generate_random_timestamp(start_date, end_date):
    """Generate random timestamp between start and end date"""
    delta = end_date - start_date
    random_seconds = random.randint(0, int(delta.total_seconds()))
    return start_date + timedelta(seconds=random_seconds)

def generate_user_behavior_profile(user_type):
    """Generate behavior profile based on user type"""
    profiles = {
        'power_user': {
            'sessions_per_week': (15, 25),
            'messages_per_session': (3, 8),
            'conversation_duration_mins': (10, 45),
            'premium_probability': 0.8
        },
        'regular_user': {
            'sessions_per_week': (5, 12),
            'messages_per_session': (2, 5),
            'conversation_duration_mins': (5, 20),
            'premium_probability': 0.4
        },
        'casual_user': {
            'sessions_per_week': (1, 4),
            'messages_per_session': (1, 3),
            'conversation_duration_mins': (2, 10),
            'premium_probability': 0.1
        },
        'churn_risk_user': {
            'sessions_per_week': (0, 2),
            'messages_per_session': (1, 2),
            'conversation_duration_mins': (1, 5),
            'premium_probability': 0.05
        }
    }
    return profiles[user_type]

def generate_realistic_message():
    """Generate realistic mock message text"""
    template = random.choice(MESSAGE_TEMPLATES)
    
    if '{feature}' in template:
        return template.format(feature=random.choice(FEATURES))
    elif '{action}' in template:
        return template.format(action=random.choice(ACTIONS))
    elif '{issue}' in template:
        return template.format(issue=random.choice(ISSUES))
    elif '{service}' in template:
        return template.format(service=random.choice(['premium', 'basic', 'enterprise']))
    elif '{topic}' in template:
        return template.format(topic=random.choice(FEATURES))
    elif '{request}' in template:
        return template.format(request=random.choice(ACTIONS))
    else:
        return template

# =============================================================================
# GENERATE USERS TABLE
# =============================================================================

print("👥 Generating users table...")

users_data = []
user_ids = [f"user_{i:05d}" for i in range(1, TOTAL_USERS + 1)]

for i, user_id in enumerate(user_ids):
    # Assign user type based on distribution
    user_type = np.random.choice(USER_TYPES, p=USER_TYPE_DISTRIBUTION)
    behavior_profile = generate_user_behavior_profile(user_type)
    
    # Registration date (spread over last 90 days)
    registration_date = generate_random_timestamp(START_DATE, END_DATE)
    
    # Determine if user is premium based on user type
    is_premium = np.random.random() < behavior_profile['premium_probability']
    
    # User demographics (optional fields with some nulls)
    country = np.random.choice(['US', 'UK', 'Canada', 'Australia', 'Germany', 'France', 'Japan', 'Brazil', None], 
                              p=[0.4, 0.15, 0.1, 0.05, 0.08, 0.07, 0.05, 0.05, 0.05])
    
    device_type = np.random.choice(['iOS', 'Android', 'Web', 'Desktop'], 
                                  p=[0.45, 0.4, 0.1, 0.05])
    
    users_data.append({
        'user_id': user_id,
        'registration_date': registration_date,
        'user_type': user_type,
        'is_premium': is_premium,
        'country': country,
        'device_type': device_type,
        'last_active_date': None  # Will be updated later
    })

users_df = pd.DataFrame(users_data)
print(f"✅ Generated {len(users_df)} users")

# =============================================================================
# GENERATE CONVERSATIONS TABLE
# =============================================================================

print("💬 Generating conversations table...")

conversations_data = []
conversation_ids = [f"conv_{i:06d}" for i in range(1, TOTAL_CONVERSATIONS + 1)]

# Track user activity for updating last_active_date
user_last_activity = {user_id: None for user_id in user_ids}

for conv_id in conversation_ids:
    # Select a random user
    user_id = random.choice(user_ids)
    user_type = users_df[users_df['user_id'] == user_id]['user_type'].iloc[0]
    behavior_profile = generate_user_behavior_profile(user_type)
    
    # Generate conversation start time
    start_time = generate_random_timestamp(START_DATE, END_DATE)
    
    # Conversation duration based on user type
    min_dur, max_dur = behavior_profile['conversation_duration_mins']
    duration_minutes = random.randint(min_dur, max_dur)
    end_time = start_time + timedelta(minutes=duration_minutes)
    
    # Update user's last activity
    if user_last_activity[user_id] is None or end_time > user_last_activity[user_id]:
        user_last_activity[user_id] = end_time
    
    conversation_category = random.choice(CONVERSATION_CATEGORIES)
    
    # Determine satisfaction score (1-5)
    base_score = random.randint(3, 5)
    # Power users tend to have higher satisfaction
    if user_type == 'power_user':
        base_score = min(5, base_score + 1)
    elif user_type == 'churn_risk_user':
        base_score = max(1, base_score - 1)
    
    conversations_data.append({
        'conversation_id': conv_id,
        'user_id': user_id,
        'start_time': start_time,
        'end_time': end_time,
        'duration_minutes': duration_minutes,
        'category': conversation_category,
        'satisfaction_score': base_score,
        'is_resolved': random.random() > 0.2  # 80% resolved
    })

conversations_df = pd.DataFrame(conversations_data)

# Update users with last active date
for user_id, last_active in user_last_activity.items():
    if last_active:
        users_df.loc[users_df['user_id'] == user_id, 'last_active_date'] = last_active

print(f"✅ Generated {len(conversations_df)} conversations")

# =============================================================================
# GENERATE MESSAGES TABLE
# =============================================================================

print("📨 Generating messages table...")

messages_data = []
message_id_counter = 1

for _, conversation in conversations_df.iterrows():
    conv_id = conversation['conversation_id']
    user_id = conversation['user_id']
    start_time = conversation['start_time']
    end_time = conversation['end_time']
    duration = conversation['duration_minutes']
    
    user_type = users_df[users_df['user_id'] == user_id]['user_type'].iloc[0]
    behavior_profile = generate_user_behavior_profile(user_type)
    
    # Number of messages in this conversation
    min_msgs, max_msgs = behavior_profile['messages_per_session']
    num_messages = random.randint(min_msgs, max_msgs)
    
    # Generate message timestamps distributed throughout conversation
    message_times = sorted([start_time + timedelta(
        seconds=random.randint(0, int(duration * 60))
    ) for _ in range(num_messages)])
    
    for i, msg_time in enumerate(message_times):
        is_user_message = i % 2 == 0  # Alternate between user and system
        
        messages_data.append({
            'message_id': f"msg_{message_id_counter:07d}",
            'conversation_id': conv_id,
            'user_id': user_id if is_user_message else 'system',
            'timestamp': msg_time,
            'message_text': generate_realistic_message(),
            'is_user_message': is_user_message,
            'message_length': len(generate_realistic_message())
        })
        message_id_counter += 1

messages_df = pd.DataFrame(messages_data)
print(f"✅ Generated {len(messages_df)} messages")

# =============================================================================
# GENERATE USER EVENTS TABLE
# =============================================================================

print("📊 Generating user events table...")

events_data = []
event_id_counter = 1

# Generate events for each user
for user_id in user_ids:
    user_type = users_df[users_df['user_id'] == user_id]['user_type'].iloc[0]
    behavior_profile = generate_user_behavior_profile(user_type)
    registration_date = users_df[users_df['user_id'] == user_id]['registration_date'].iloc[0]
    
    # Number of sessions for this user
    min_sessions, max_sessions = behavior_profile['sessions_per_week']
    weeks_since_registration = max(1, (END_DATE - registration_date).days // 7)
    total_sessions = random.randint(
        min_sessions * weeks_since_registration, 
        max_sessions * weeks_since_registration
    )
    
    # Event distribution by type
    event_weights = [0.25, 0.20, 0.10, 0.10, 0.08, 0.07, 0.10, 0.05, 0.05]
    
    for _ in range(total_sessions):
        # Generate session events
        session_start = generate_random_timestamp(registration_date, END_DATE)
        num_session_events = random.randint(3, 12)
        
        for j in range(num_session_events):
            event_time = session_start + timedelta(minutes=j*2 + random.randint(0, 5))
            event_type = np.random.choice(EVENT_TYPES, p=event_weights)
            
            # Add metadata based on event type
            metadata = {}
            if event_type == 'premium_feature_used':
                metadata['feature'] = random.choice(FEATURES)
            elif event_type == 'settings_changed':
                metadata['setting'] = random.choice(['theme', 'language', 'notifications'])
            
            events_data.append({
                'event_id': f"event_{event_id_counter:08d}",
                'user_id': user_id,
                'event_type': event_type,
                'timestamp': event_time,
                'session_id': f"session_{user_id}_{session_start.strftime('%Y%m%d_%H%M')}",
                'metadata': str(metadata) if metadata else None
            })
            event_id_counter += 1

# Trim to exact total if needed
events_data = events_data[:TOTAL_EVENTS]
events_df = pd.DataFrame(events_data)
print(f"✅ Generated {len(events_df)} user events")

# =============================================================================
# DATA VALIDATION & QUALITY CHECKS
# =============================================================================

print("🔍 Performing data validation...")

# Basic validation checks
validation_checks = {
    "Unique users": users_df['user_id'].nunique() == TOTAL_USERS,
    "Unique conversations": conversations_df['conversation_id'].nunique() == TOTAL_CONVERSATIONS,
    "Unique messages": messages_df['message_id'].nunique() == len(messages_df),
    "Unique events": events_df['event_id'].nunique() == len(events_df),
    "No null user IDs": not users_df['user_id'].isnull().any(),
    "Valid date ranges": all(conversations_df['end_time'] >= conversations_df['start_time']),
    "User type distribution": abs(users_df['user_type'].value_counts(normalize=True) - USER_TYPE_DISTRIBUTION).max() < 0.05
}

for check_name, check_result in validation_checks.items():
    status = "✅ PASS" if check_result else "❌ FAIL"
    print(f"{status} {check_name}")

# Data quality metrics
print("\n📈 Data Quality Metrics:")
print(f"Users by type:\n{users_df['user_type'].value_counts()}")
print(f"Premium users: {users_df['is_premium'].sum()} ({users_df['is_premium'].mean():.1%})")
print(f"Average conversation duration: {conversations_df['duration_minutes'].mean():.1f} minutes")
print(f"Average messages per conversation: {len(messages_df) / len(conversations_df):.1f}")
print(f"Average events per user: {len(events_df) / len(users_df):.1f}")

# =============================================================================
# EXPORT DATA
# =============================================================================

print("\n💾 Exporting data to CSV files...")

# Export each table to CSV
users_df.to_csv('mock_users.csv', index=False)
conversations_df.to_csv('mock_conversations.csv', index=False)
messages_df.to_csv('mock_messages.csv', index=False)
events_df.to_csv('mock_user_events.csv', index=False)

print("✅ Data exported successfully!")
print("\n📁 Generated Files:")
print("  - mock_users.csv")
print("  - mock_conversations.csv")
print("  - mock_messages.csv")
print("  - mock_user_events.csv")

# =============================================================================
# SAMPLE DATA PREVIEW
# =============================================================================

print("\n👀 Sample Data Preview:")

print("\n--- Users Sample ---")
print(users_df.head(3))

print("\n--- Conversations Sample ---")
print(conversations_df.head(3))

print("\n--- Messages Sample ---")
print(messages_df.head(3))

print("\n--- Events Sample ---")
print(events_df.head(3))

print(f"\n🎉 Mock dataset generation completed successfully!")
print(f"📊 Total records generated:")
print(f"   Users: {len(users_df):,}")
print(f"   Conversations: {len(conversations_df):,}")
print(f"   Messages: {len(messages_df):,}")
print(f"   Events: {len(events_df):,}")

🚀 Starting Mock Dataset Generation...
✅ Configuration loaded successfully
👥 Generating users table...
✅ Generated 1000 users
💬 Generating conversations table...
✅ Generated 5000 conversations
📨 Generating messages table...
✅ Generated 15358 messages
📊 Generating user events table...
✅ Generated 75000 user events
🔍 Performing data validation...
✅ PASS Unique users
✅ PASS Unique conversations
✅ PASS Unique messages
✅ PASS Unique events
✅ PASS No null user IDs
✅ PASS Valid date ranges
❌ FAIL User type distribution

📈 Data Quality Metrics:
Users by type:
user_type
regular_user       442
casual_user        312
power_user         142
churn_risk_user    104
Name: count, dtype: int64
Premium users: 327 (32.7%)
Average conversation duration: 11.4 minutes
Average messages per conversation: 3.1
Average events per user: 75.0

💾 Exporting data to CSV files...
✅ Data exported successfully!

📁 Generated Files:
  - mock_users.csv
  - mock_conversations.csv
  - mock_messages.csv
  - mock_user_events.cs